# Laboratorio Dirigido N.° 04 — Modelado relacional
**Curso:** Fundamentos de Gestión de Datos · TECSUP 2026-II
**Docente:** Pilar Rocío Sayán Mejía
**Estudiante:** _(escribe aquí tu nombre)_
**Caso:** Panadería Dulce Migaja

> Los datos de Panadería Dulce Migaja son **simulados**. El cálculo del ingreso supone **precios constantes** (`cantidad × precio` del producto).

## 1. Reglas de negocio y modelo relacional

**Reglas de negocio**
- Cada producto pertenece a **una** categoría; una categoría agrupa **muchos** productos (1:N).
- Cada venta la realiza **un** cliente; un cliente puede hacer **muchas** ventas (1:N).
- Cada venta se registra en **un** local; un local registra **muchas** ventas (1:N).
- Una venta puede incluir **muchos** productos y un producto puede aparecer en **muchas** ventas (**N:M**), por eso se resuelve con la tabla intermedia `detalle_venta`.

**Modelo relacional** (PK subrayada mentalmente como `PK`, claves foráneas como `FK`)
- `categoria(id_categoria PK, nombre)`
- `producto(id_producto PK, nombre, precio, id_categoria FK → categoria)`
- `local(id_local PK, nombre, distrito)`
- `cliente(id_cliente PK, nombre, telefono)`
- `venta(id_venta PK, fecha, id_cliente FK → cliente, id_local FK → local)`
- `detalle_venta(id_venta FK → venta, id_producto FK → producto, cantidad)` con **PK compuesta** `(id_venta, id_producto)`

La **PK** identifica cada fila de forma única. La **FK** relaciona tablas y **puede repetirse en el lado N** (por ejemplo, el mismo `id_cliente` aparece en varias ventas).

## 2. Normalización (de tabla sin normalizar a 3FN)

**Tabla inicial (sin normalizar):** una fila por venta, con todos los productos en una sola celda.

| id_venta | fecha | cliente | telefono | local | distrito | productos |
|---|---|---|---|---|---|---|
| 1 | 2026-10-01 | Ana Quispe | 958111222 | Local Centro | Cercado | Pan francés x10, Torta de chocolate x1 |

**1FN — valores atómicos.** La columna `productos` contiene varios valores en una celda (grupo repetido), así que **no cumple 1FN**. Se separa en una fila por producto vendido. La clave pasa a ser **compuesta**: `(id_venta, id_producto)`.

`ventas_1fn(id_venta, id_producto, fecha, id_cliente, cliente, telefono, id_local, local, distrito, producto, categoria, precio, cantidad)`

**2FN — sin dependencias parciales.** Con la clave `(id_venta, id_producto)`:
- `id_venta → fecha, id_cliente, id_local` (depende solo de **parte** de la clave)
- `id_producto → producto, precio, id_categoria` (depende solo de **parte** de la clave)
- Solo `cantidad` depende de la clave completa.

Se separan: `venta(id_venta, fecha, id_cliente, id_local)`, `producto(id_producto, nombre, precio, id_categoria)` y `detalle_venta(id_venta, id_producto, cantidad)`.

**3FN — sin dependencias transitivas.**
- En `venta`: `id_venta → id_cliente → nombre, telefono` y `id_venta → id_local → nombre, distrito` son transitivas → se crean `cliente` y `local`.
- En `producto`: `id_producto → id_categoria → nombre_categoria` es transitiva → se crea `categoria`.

**Resultado:** las 6 tablas del modelo están en 3FN. La justificación se basa en la **clave y las dependencias funcionales**, no en que un texto se repita: por ejemplo, el mismo `id_local` repetido en `venta` es correcto porque es una FK en el lado N.

## 3. Creación de las tablas en SQLite

In [1]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("PRAGMA foreign_keys = ON;")   # activa la integridad referencial en SQLite
cur = conn.cursor()

def mostrar(sql, params=()):
    """Ejecuta una consulta y muestra el resultado como tabla."""
    cur.execute(sql, params)
    cols = [d[0] for d in cur.description]
    filas = cur.fetchall()
    datos = [cols] + [[("" if v is None else str(v)) for v in f] for f in filas]
    anchos = [max(len(fila[i]) for fila in datos) for i in range(len(cols))]
    linea = lambda f: " | ".join(v.ljust(anchos[i]) for i, v in enumerate(f))
    print(linea(cols))
    print("-+-".join("-" * a for a in anchos))
    for f in datos[1:]:
        print(linea(f))
    print(f"({len(filas)} filas)")

print("Conexión creada. foreign_keys =", conn.execute("PRAGMA foreign_keys").fetchone()[0])

Conexión creada. foreign_keys = 1


In [2]:
cur.executescript("""
CREATE TABLE categoria (
    id_categoria INTEGER PRIMARY KEY,
    nombre       TEXT NOT NULL UNIQUE
);

CREATE TABLE producto (
    id_producto  INTEGER PRIMARY KEY,
    nombre       TEXT NOT NULL,
    precio       REAL NOT NULL CHECK (precio > 0),
    id_categoria INTEGER NOT NULL REFERENCES categoria(id_categoria)
);

CREATE TABLE local (
    id_local INTEGER PRIMARY KEY,
    nombre   TEXT NOT NULL,
    distrito TEXT NOT NULL
);

CREATE TABLE cliente (
    id_cliente INTEGER PRIMARY KEY,
    nombre     TEXT NOT NULL,
    telefono   TEXT
);

CREATE TABLE venta (
    id_venta   INTEGER PRIMARY KEY,
    fecha      TEXT NOT NULL,
    id_cliente INTEGER NOT NULL REFERENCES cliente(id_cliente),
    id_local   INTEGER NOT NULL REFERENCES local(id_local)
);

-- Tabla intermedia que resuelve la relación N:M entre venta y producto
CREATE TABLE detalle_venta (
    id_venta    INTEGER NOT NULL REFERENCES venta(id_venta),
    id_producto INTEGER NOT NULL REFERENCES producto(id_producto),
    cantidad    INTEGER NOT NULL CHECK (cantidad > 0),
    PRIMARY KEY (id_venta, id_producto)
);
""")
mostrar("SELECT name AS tabla FROM sqlite_master WHERE type='table' ORDER BY name")

tabla        
-------------
categoria    
cliente      
detalle_venta
local        
producto     
venta        
(6 filas)


## 4. Carga de datos (INSERT INTO)

In [3]:
cur.executemany("INSERT INTO categoria VALUES (?,?)", [
    (1, "Panes"), (2, "Tortas"), (3, "Bebidas"), (4, "Bocaditos")])

cur.executemany("INSERT INTO producto VALUES (?,?,?,?)", [
    (1, "Pan francés",        0.30, 1),
    (2, "Pan integral",       0.50, 1),
    (3, "Torta de chocolate", 45.00, 2),
    (4, "Pie de manzana",     30.00, 2),
    (5, "Café pasado",         5.00, 3),
    (6, "Chicha morada",       4.00, 3),
    (7, "Empanada de carne",   6.00, 4),
    (8, "Alfajor",             3.50, 4)])

cur.executemany("INSERT INTO local VALUES (?,?,?)", [
    (1, "Local Centro",   "Cercado"),
    (2, "Local Yanahuara", "Yanahuara"),
    (3, "Local Cayma",    "Cayma")])

cur.executemany("INSERT INTO cliente VALUES (?,?,?)", [
    (1, "Ana Quispe",    "958111222"),
    (2, "Luis Mamani",   "959333444"),
    (3, "Rosa Condori",  "960555666"),
    (4, "Jorge Flores",  "961777888"),
    (5, "Carmen Huamán", "962999000")])

cur.executemany("INSERT INTO venta VALUES (?,?,?,?)", [
    (1, "2026-10-01", 1, 1),
    (2, "2026-10-01", 2, 2),
    (3, "2026-10-02", 3, 1),
    (4, "2026-10-02", 1, 3),
    (5, "2026-10-03", 4, 2),
    (6, "2026-10-03", 2, 1),
    (7, "2026-10-04", 3, 3),
    (8, "2026-10-04", 1, 2)])

cur.executemany("INSERT INTO detalle_venta VALUES (?,?,?)", [
    (1, 1, 10), (1, 3, 1),
    (2, 2, 6),  (2, 5, 2),
    (3, 7, 3),  (3, 6, 2), (3, 8, 4),
    (4, 4, 1),  (4, 5, 1),
    (5, 1, 20), (5, 8, 6),
    (6, 3, 1),  (6, 7, 2),
    (7, 1, 15), (7, 2, 10), (7, 6, 1),
    (8, 5, 3),  (8, 8, 2)])
conn.commit()

for t in ["categoria", "producto", "local", "cliente", "venta", "detalle_venta"]:
    n = cur.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"{t:<14} {n} filas")

categoria      4 filas
producto       8 filas
local          3 filas
cliente        5 filas
venta          8 filas
detalle_venta  18 filas


## 5. Comprobación del modelo con JOIN

In [4]:
mostrar("""
SELECT v.id_venta, v.fecha, c.nombre AS cliente, l.nombre AS local,
       p.nombre AS producto, d.cantidad, p.precio,
       ROUND(d.cantidad * p.precio, 2) AS subtotal
FROM venta v
JOIN cliente c       ON c.id_cliente  = v.id_cliente
JOIN local l         ON l.id_local    = v.id_local
JOIN detalle_venta d ON d.id_venta    = v.id_venta
JOIN producto p      ON p.id_producto = d.id_producto
ORDER BY v.id_venta, p.nombre
""")

id_venta | fecha      | cliente      | local           | producto           | cantidad | precio | subtotal
---------+------------+--------------+-----------------+--------------------+----------+--------+---------
1        | 2026-10-01 | Ana Quispe   | Local Centro    | Pan francés        | 10       | 0.3    | 3.0     
1        | 2026-10-01 | Ana Quispe   | Local Centro    | Torta de chocolate | 1        | 45.0   | 45.0    
2        | 2026-10-01 | Luis Mamani  | Local Yanahuara | Café pasado        | 2        | 5.0    | 10.0    
2        | 2026-10-01 | Luis Mamani  | Local Yanahuara | Pan integral       | 6        | 0.5    | 3.0     
3        | 2026-10-02 | Rosa Condori | Local Centro    | Alfajor            | 4        | 3.5    | 14.0    
3        | 2026-10-02 | Rosa Condori | Local Centro    | Chicha morada      | 2        | 4.0    | 8.0     
3        | 2026-10-02 | Rosa Condori | Local Centro    | Empanada de carne  | 3        | 6.0    | 18.0    
4        | 2026-10-02 | Ana Quispe   

**Interpretación:** el JOIN reconstruye la información completa de cada venta a partir de las tablas normalizadas. Que todas las filas se unan correctamente demuestra que las FK apuntan a PK existentes. El mismo `id_venta` aparece en varias filas porque una venta tiene varios productos (lado N del detalle).

### 5.1 Prueba de integridad referencial

In [5]:
try:
    cur.execute("INSERT INTO venta VALUES (99, '2026-10-05', 999, 1)")  # cliente 999 no existe
except sqlite3.IntegrityError as e:
    print("Inserción rechazada:", e)

try:
    cur.execute("INSERT INTO detalle_venta VALUES (1, 1, 5)")  # (venta 1, producto 1) ya existe
except sqlite3.IntegrityError as e:
    print("Inserción rechazada:", e)

Inserción rechazada: FOREIGN KEY constraint failed
Inserción rechazada: UNIQUE constraint failed: detalle_venta.id_venta, detalle_venta.id_producto


**Interpretación:** SQLite rechaza una venta con un cliente inexistente (violación de FK) y una fila duplicada en el detalle (violación de la PK compuesta). Así el modelo impide registros huérfanos o duplicados.

## 6. Retos individuales

### Reto 1 — Relación N:M (venta ↔ producto)
Una venta tiene varios productos y un producto aparece en varias ventas. Se comprueba con la tabla intermedia `detalle_venta`.

In [6]:
print("Productos por venta (una venta -> muchos productos):")
mostrar("""
SELECT v.id_venta, COUNT(d.id_producto) AS n_productos,
       GROUP_CONCAT(p.nombre, ', ') AS productos
FROM venta v
JOIN detalle_venta d ON d.id_venta = v.id_venta
JOIN producto p      ON p.id_producto = d.id_producto
GROUP BY v.id_venta
""")
print()
print("Ventas por producto (un producto -> muchas ventas):")
mostrar("""
SELECT p.nombre AS producto, COUNT(d.id_venta) AS n_ventas,
       GROUP_CONCAT(d.id_venta, ', ') AS ventas
FROM producto p
JOIN detalle_venta d ON d.id_producto = p.id_producto
GROUP BY p.id_producto
ORDER BY n_ventas DESC
""")

Productos por venta (una venta -> muchos productos):
id_venta | n_productos | productos                                
---------+-------------+------------------------------------------
1        | 2           | Pan francés, Torta de chocolate          
2        | 2           | Pan integral, Café pasado                
3        | 3           | Chicha morada, Empanada de carne, Alfajor
4        | 2           | Pie de manzana, Café pasado              
5        | 2           | Pan francés, Alfajor                     
6        | 2           | Torta de chocolate, Empanada de carne    
7        | 3           | Pan francés, Pan integral, Chicha morada 
8        | 2           | Café pasado, Alfajor                     
(8 filas)

Ventas por producto (un producto -> muchas ventas):
producto           | n_ventas | ventas 
-------------------+----------+--------
Alfajor            | 3        | 3, 5, 8
Café pasado        | 3        | 2, 4, 8
Pan francés        | 3        | 1, 5, 7
Empanada de ca

**Interpretación:** la venta 3 incluye tres productos y, al mismo tiempo, el pan francés aparece en tres ventas distintas. Esto confirma la cardinalidad N:M. No se puede representar con una sola FK, por eso `detalle_venta` tiene dos FK y una PK compuesta `(id_venta, id_producto)`, además del atributo propio de la relación: `cantidad`.

### Reto 2 — Registro de una venta nueva
Se registra la venta 9: la clienta Carmen Huamán (id 5) compra en el Local Cayma (id 3) una torta de chocolate y 4 cafés.

In [7]:
cur.execute("INSERT INTO venta VALUES (9, '2026-10-05', 5, 3)")
cur.executemany("INSERT INTO detalle_venta VALUES (?,?,?)", [(9, 3, 1), (9, 5, 4)])
conn.commit()

mostrar("""
SELECT v.id_venta, v.fecha, c.nombre AS cliente, l.nombre AS local,
       p.nombre AS producto, d.cantidad,
       ROUND(d.cantidad * p.precio, 2) AS subtotal
FROM venta v
JOIN cliente c       ON c.id_cliente  = v.id_cliente
JOIN local l         ON l.id_local    = v.id_local
JOIN detalle_venta d ON d.id_venta    = v.id_venta
JOIN producto p      ON p.id_producto = d.id_producto
WHERE v.id_venta = 9
""")
print()
mostrar("""SELECT ROUND(SUM(d.cantidad * p.precio), 2) AS total_venta_9
FROM detalle_venta d JOIN producto p ON p.id_producto = d.id_producto
WHERE d.id_venta = 9""")

id_venta | fecha      | cliente       | local       | producto           | cantidad | subtotal
---------+------------+---------------+-------------+--------------------+----------+---------
9        | 2026-10-05 | Carmen Huamán | Local Cayma | Torta de chocolate | 1        | 45.0    
9        | 2026-10-05 | Carmen Huamán | Local Cayma | Café pasado        | 4        | 20.0    
(2 filas)

total_venta_9
-------------
65.0         
(1 filas)


**Interpretación:** primero se inserta la cabecera en `venta` y luego sus líneas en `detalle_venta`; el orden importa porque el detalle depende de que la venta exista (FK). El JOIN confirma que la venta quedó vinculada a una clienta y un local válidos. Su total es S/ 65.00.

### Reto 3 — Ventas por local

In [8]:
mostrar("""
SELECT l.nombre AS local,
       COUNT(DISTINCT v.id_venta)              AS n_ventas,
       ROUND(SUM(d.cantidad * p.precio), 2)    AS ingreso
FROM local l
JOIN venta v         ON v.id_local    = l.id_local
JOIN detalle_venta d ON d.id_venta    = v.id_venta
JOIN producto p      ON p.id_producto = d.id_producto
GROUP BY l.id_local
ORDER BY ingreso DESC
""")

local           | n_ventas | ingreso
----------------+----------+--------
Local Centro    | 3        | 145.0  
Local Cayma     | 3        | 113.5  
Local Yanahuara | 3        | 62.0   
(3 filas)


**Interpretación:** se usa `COUNT(DISTINCT id_venta)` porque, tras el JOIN con el detalle, cada venta aparece una vez por producto; sin `DISTINCT` se contarían líneas, no ventas. Los tres locales tienen 3 ventas, pero Local Centro factura más del doble que Yanahuara porque sus ventas incluyen productos de mayor precio (tortas). El local con mayor ingreso no es necesariamente el más **rentable**: no se conocen sus costos (alquiler, personal, insumos).

### Reto 4 — Ingreso por categoría

In [9]:
mostrar("""
SELECT c.nombre AS categoria,
       SUM(d.cantidad)                         AS unidades,
       ROUND(SUM(d.cantidad * p.precio), 2)    AS ingreso
FROM categoria c
JOIN producto p      ON p.id_categoria = c.id_categoria
JOIN detalle_venta d ON d.id_producto  = p.id_producto
GROUP BY c.id_categoria
ORDER BY ingreso DESC
""")

categoria | unidades | ingreso
----------+----------+--------
Tortas    | 4        | 165.0  
Bocaditos | 17       | 72.0   
Bebidas   | 13       | 62.0   
Panes     | 61       | 21.5   
(4 filas)


**Interpretación:** Tortas genera el mayor ingreso con pocas unidades, mientras que Panes vende muchas unidades pero aporta poco ingreso por su precio bajo. Volumen e ingreso no van de la mano. Además, más ingreso no demuestra más rentabilidad, ya que no se tiene el costo de cada producto; y el cálculo supone precios constantes.

### Reto 5 — Compras por cliente

In [10]:
mostrar("""
SELECT c.nombre AS cliente,
       COUNT(DISTINCT v.id_venta)              AS n_compras,
       ROUND(SUM(d.cantidad * p.precio), 2)    AS total_gastado
FROM cliente c
LEFT JOIN venta v         ON v.id_cliente  = c.id_cliente
LEFT JOIN detalle_venta d ON d.id_venta    = v.id_venta
LEFT JOIN producto p      ON p.id_producto = d.id_producto
GROUP BY c.id_cliente
ORDER BY total_gastado DESC
""")

cliente       | n_compras | total_gastado
--------------+-----------+--------------
Ana Quispe    | 3         | 105.0        
Luis Mamani   | 2         | 70.0         
Carmen Huamán | 1         | 65.0         
Rosa Condori  | 2         | 53.5         
Jorge Flores  | 1         | 27.0         
(5 filas)


**Interpretación:** se usa `LEFT JOIN` para que aparezcan todos los clientes, aunque no tuvieran compras. El mismo `id_cliente` se repite en varias ventas porque es FK en el lado N de la relación cliente–venta. Quien compra más veces no siempre es quien más gasta: depende de qué productos lleva.

## 7. Respuestas a la guía

- **¿Qué identifica la PK?** Identifica de forma única cada fila; no se repite ni puede ser nula. En `detalle_venta` la PK es compuesta `(id_venta, id_producto)`.
- **¿Por qué la FK puede repetirse?** Porque está en el lado N de una relación 1:N: un cliente tiene muchas ventas, entonces su `id_cliente` aparece en varias filas de `venta`.
- **¿Cómo se resuelve una relación N:M?** Con una tabla intermedia (`detalle_venta`) que contiene las FK de ambas tablas y los atributos propios de la relación (`cantidad`).
- **¿Cómo se justifica la normalización?** Con la clave y las dependencias funcionales: 1FN elimina valores no atómicos, 2FN elimina dependencias parciales de una clave compuesta y 3FN elimina dependencias transitivas. Que un texto se repita no basta para afirmar que se incumple una forma normal.
- **¿Para qué sirve `PRAGMA foreign_keys = ON`?** En SQLite la integridad referencial está desactivada por defecto; este comando la activa para que se rechacen FK inválidas.

## 8. Conclusiones

1. **La normalización hasta 3FN reduce redundancia y anomalías.** Al separar cliente, local, producto y categoría, cada dato se guarda una sola vez y se actualiza en un solo lugar; la decisión se justificó con claves y dependencias funcionales, no por la repetición de textos.
2. **PK y FK garantizan la integridad del modelo.** SQLite rechazó una venta con un cliente inexistente y un detalle duplicado, y los JOIN reconstruyeron cada venta completa, lo que comprueba que las relaciones 1:N y N:M están bien implementadas.
3. **Las consultas apoyan decisiones, pero con límites.** Se identificaron los locales, categorías y clientes con mayor ingreso; sin embargo, mayor ingreso no demuestra mayor rentabilidad, los datos son simulados y el cálculo supone precios constantes, por lo que para conclusiones de negocio se necesitarían costos y precios históricos.

In [11]:
conn.close()
print("Conexión cerrada.")

Conexión cerrada.
